# 초구 예측 — model_01

- 출발점: 베이스라인 모델 구조 그대로 (구종 B4 → 위치 L2 정규분포 적분)
- 데이터·채점·저장은 `fp_common`, 모델 함수만 이 노트북에
- 1단계: 지금 구조에 맞게 파라미터 전부 다시 고르기 (2019로만 고름, 2020은 compare에서 확인)
  - 고르는 방식: 거친 격자로 조합 전부 → 1등 주변을 세밀한 격자로 다시 조합

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc  # 데이터·24칸 정의·채점·저장 공통 모듈

MODEL = "model_01"

m, names = fc.load_data()  # m: 전체 투구 (24칸·높이 비율 포함), names: 투수 코드 → 이름
fp, y_type, y_zone, val, test = fc.first_pitches(m)  # fp: 초구(0-0)만, y_*: 정답 번호, val/test: 2019/2020 행 표시 (T/F)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]  # 2019 검증 / 2020 평가 초구
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")

초구 50,709개 (검증 13,458 / 평가 12,615)


## 1. 파라미터 (출발값)

- 베이스라인에서 고른 값으로 시작 → 아래 선택 셀에서 고른 값으로 덮어씀
- 반감기 `None` = 반감 없이 전날까지 누적, `hand_k` `None` = 타자 손 구분 안 함

In [2]:
# ① 구종 (B4): 투수 초구 비율을 리그 쪽으로 alpha만큼, 다시 투수×타자 손 비율 쪽으로 hand_k만큼 당김
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = 40, 1.0, 20  # 반감기(공 수), 리그 쪽 당김, 타자 손 쪽 당김
# ② 위치 (L2): 리그 구종×손 → 투수×구종(K_TYPE) → 투수×구종×손(K_HAND) 순서로 당김
LOC_HL, K_TYPE, K_HAND = 300, 300, 300  # 반감기(공 수), 투수×구종 / 투수×구종×타자 손 당김

## 2. ① 구종

- 리그 초구 비율 → 투수 초구 비율(전날까지, 반감기) → 투수×타자 손 비율 순서로 당김
- 당김 = `(투수 합계 + k × 상위 비율) / (투수 공 수 + k)` → 공이 적으면 상위 비율에 가깝고, 많으면 투수 비율에 가까움

In [3]:
def league_type(year):  # 지난 시즌까지 리그 초구 구종 비율 (9개, 합 1)
    return fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()  # 구종 원-핫 열의 평균 = 비율

def type_online(q, year, half_life, alpha, hand_k):  # 초구 구종 확률 (n, 9): B4, hand_k=None이면 B3, q: 예측할 초구
    # 투수별 전날까지 구종 원-핫 (반감기 가중) 합계 S, 공 수 N -> 같은 날 공은 제외됨 (lookup)
    S, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, half_life), q, ["pitcher_code"], CLASSES)
    P = fc.shrink(S, N[:, :1], league_type(year), alpha)  # 원-핫이라 9개 열의 N이 같음 -> 첫 열(= 공 수)만
    if hand_k:  # 그 결과에서 다시 투수×같은 손/반대 손 비율 쪽으로 hand_k만큼
        keys = ["pitcher_code", "same_side"]
        S2, N2 = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life), q, keys, CLASSES)
        P = fc.shrink(S2, N2[:, :1], P, hand_k)
    return P

### 2-1. 구종 파라미터 선택 (2019)

- 반감기·alpha·hand_k를 함께 고름
- 기준: 2019 구종 log loss (top2는 참고)
- 순서: 거친 격자 조합 전부 → 1등 값 주변 세밀한 격자 조합 전부

In [4]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN → None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로"""
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN → None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def score_type(half_life, alpha, hand_k):  # 구종 파라미터 한 조합의 2019 채점 (log loss, top1~3)
    return fc.prob_metrics(type_online(q_val, fc.VAL_YEAR, half_life, alpha, hand_k), y_type[val])

In [5]:
# 거친 격자
grid_type = {"half_life": [None, 400, 200, 100, 50, 25, 12],  # None = 누적
             "alpha": [0.1, 0.3, 1, 3, 10, 30],
             "hand_k": [None, 3, 10, 30, 100, 300, 1000]}  # None = 손 구분 안 함
res_type = grid_search(score_type, grid_type)
display(res_type.head(10).round(4))  # 상위 10개 조합
for k, s in marginal(res_type, grid_type).items():  # 파라미터별 민감도
    print(k, s.round(4).to_dict())
best_type = best_of(res_type, grid_type)
print("거친 격자 1등:", best_type)

,half_life,alpha,hand_k,logloss,top1,top2,top3
0,25.0,3.0,10.0,1.0988,0.5272,0.8148,0.9313
1,25.0,10.0,3.0,1.0991,0.5253,0.8149,0.9337
2,25.0,30.0,3.0,1.0997,0.5259,0.8153,0.9338
3,25.0,10.0,10.0,1.1001,0.5284,0.8149,0.9331
4,25.0,1.0,10.0,1.1008,0.5268,0.8148,0.9310
5,25.0,3.0,3.0,1.1008,0.5251,0.8150,0.9339
6,50.0,10.0,10.0,1.1021,0.5295,0.8124,0.9314
7,50.0,30.0,10.0,1.1034,0.5291,0.8123,0.9316
8,50.0,3.0,10.0,1.1036,0.5301,0.8122,0.9322
9,50.0,30.0,3.0,1.1036,0.5297,0.8119,0.9325


half_life {'None': 1.1429, 400: 1.1322, 200: 1.1241, 100: 1.1129, 50: 1.1021, 25: 1.0988, 12: 1.1111}
alpha {0.1: 1.1076, 0.3: 1.1042, 1.0: 1.1008, 3.0: 1.0988, 10.0: 1.0991, 30.0: 1.0997}
hand_k {'None': 1.1531, 3: 1.0991, 10: 1.0988, 30: 1.1054, 100: 1.1187, 300: 1.1352, 1000: 1.1464}
거친 격자 1등: {'half_life': 25.0, 'alpha': 3.0, 'hand_k': 10.0}


### 🥇1등🥇
#### 반감 25구, alpha 3, hand_k 10 -> 2019 log loss 1.0988 
#### (베이스라인 40 / 1 / 20: 1.1037)
- 반감기가 가장 민감: 누적 1.143 → 25구 1.099, 12구에서 다시 1.111 -> 최근 1경기 정도(선발 초구 약 25개)를 가장 크게 반영
- hand_k: 손 구분 없음 1.153으로 가장 나쁨 -> 손 구분은 꼭 필요, 3–10으로 투수×손 비율을 강하게 따라감
- alpha: 3–30에서 1.0988–1.0997로 거의 같음 -> 둔감, 베이스라인 alpha=1은 조금 손해
- 세 파라미터 모두 1등이 후보 범위 안쪽 -> 범위 넓힐 필요 없음

In [6]:
# 세밀한 격자
grid_type_f = {k: finer(grid_type[k], best_type[k]) for k in grid_type}
print("세밀한 후보:", grid_type_f)
res_type_f = grid_search(score_type, grid_type_f)
display(res_type_f.head(10).round(4))
for k, s in marginal(res_type_f, grid_type_f).items():
    print(k, s.round(4).to_dict())
best_type_f = best_of(res_type_f, grid_type_f)

# 선택값 적용 (실전 예측은 이 값 사용)
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = best_type_f["half_life"], best_type_f["alpha"], best_type_f["hand_k"]
base_ll = score_type(40, 1.0, 20)["logloss"]  # 베이스라인 값 (2019)
print(f"선택: half_life={TYPE_HL}, alpha={TYPE_ALPHA}, hand_k={TYPE_HAND_K} "
      f"| 2019 log loss {res_type_f['logloss'].iloc[0]:.4f} (베이스라인 {base_ll:.4f})")

세밀한 후보: {'half_life': [12.0, 17.3, 25.0, 35.4, 50.0], 'alpha': [1.0, 1.73, 3.0, 5.48, 10.0], 'hand_k': [3.0, 5.48, 10.0, 17.3, 30.0]}


,half_life,alpha,hand_k,logloss,top1,top2,top3
0,25.0,5.48,5.48,1.0982,0.5258,0.8157,0.9331
1,25.0,10.00,5.48,1.0983,0.5262,0.8159,0.9332
2,25.0,3.00,5.48,1.0988,0.5254,0.8156,0.9331
3,25.0,3.00,10.00,1.0988,0.5272,0.8148,0.9313
4,25.0,5.48,10.00,1.0988,0.5273,0.8153,0.9310
5,35.4,5.48,10.00,1.0989,0.5292,0.8140,0.9322
6,35.4,10.00,5.48,1.0990,0.5291,0.8139,0.9323
7,25.0,10.00,3.00,1.0991,0.5253,0.8149,0.9337
8,35.4,10.00,10.00,1.0991,0.5293,0.8136,0.9322
9,35.4,5.48,5.48,1.0995,0.5290,0.8133,0.9327


half_life {12.0: 1.1105, 17.3: 1.1015, 25.0: 1.0982, 35.4: 1.0989, 50.0: 1.1021}
alpha {1.0: 1.1008, 1.73: 1.0996, 3.0: 1.0988, 5.48: 1.0982, 10.0: 1.0983}
hand_k {3.0: 1.0991, 5.48: 1.0982, 10.0: 1.0988, 17.3: 1.1005, 30.0: 1.1043}
선택: half_life=25.0, alpha=5.48, hand_k=5.48 | 2019 log loss 1.0982 (베이스라인 1.1037)


#### 선택: 반감 25구, alpha 5.48, hand_k 5.48 -> 2019 log loss 1.0982 
#### (거친 격자 1등 대비 −0.0006, 베이스라인 대비 −0.0055)
- 세밀한 격자 개선은 0.0006뿐 -> 거친 격자로 이미 거의 최적, 이 근처는 평평함
- alpha 5.48 vs 10 차이 0.0001 -> 정확한 값에 크게 의존하지 않음 -> 2019 과적합 걱정 작음

## 3. ② 위치

- 리그 구종×타자 손 → 투수×구종 → 투수×구종×타자 손 순서로 위치 모멘트를 당김 → 2차원 정규분포 → 24칸 적분

In [12]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]

def loc_league(q, year):  # 지난 시즌까지 리그 구종×타자 손 평균 (조합이 없으면 리그 전체 평균)
    lg = fp[fp["GYEAR"] < year].groupby(["pitch_type", "same_side"])[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[["pitch_type", "same_side"]]))  # 공마다 그 구종×손 행
    return pr.fillna(fp.loc[fp["GYEAR"] < year, L_COLS].mean()).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year)  # 출발: 리그 구종×손
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포 (중심 x, z, 분산 x, z, 공분산)
    # 투수가 안 던진 구종이면 투수 기록이 0 -> 리그 구종×손 평균으로 예측
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand), qt)  # 모멘트 → 정규분포 (높이는 그 타자 존으로 cm 환산)
    return fc.zone_from_gauss(g, qt), g  # 정규분포를 24칸 위에서 적분 (좌우·높이 독립 가정)

### 3-1. 위치 파라미터 선택 (2019)

- 반감기, k_type, k_hand를 함께 고름 (베이스라인은 반감기 누적·300 두 가지만, 300이 범위 끝)
- 기준: 2019 24칸 log loss (실제 구종 기준 = 위치 모델 자체 실력)
  - H/S/C log loss, 좌표 NLL은 참고
- 순서: 거친 격자 → 1등 주변 세밀한 격자

In [13]:
def score_loc(half_life, k_type, k_hand):  # 위치 파라미터 한 조합의 2019 채점: 24칸 log loss·top1·H/S/C + 좌표 NLL·평균거리
    g = fc.gauss(loc_online(q_val, fc.VAL_YEAR, half_life, k_type, k_hand), q_val)  # 실제 구종 기준 정규분포
    z = fc.zone_metrics(fc.zone_from_gauss(g, q_val), y_zone[val])  # 정규분포를 24칸 위에서 적분해 채점
    c = fc.coord_metrics(g, q_val)  # 좌표 채점
    return {"logloss": z["logloss"], "top1": z["top1"], "H/S/C logloss": z["H/S/C logloss"],
            "H/S/C 정확도": z["H/S/C 정확도"], "좌표 NLL": c["NLL"], "평균거리 cm": c["평균거리 cm"]}

In [14]:
# 거친 격자
grid_loc = {"half_life": [None, 1000, 300, 100, 50],  # None = 누적
            "k_type": [10, 30, 100, 300, 1000, 3000],  # 리그 구종×손 → 투수×구종 쪽으로 당기는 강도
            "k_hand": [10, 30, 100, 300, 1000, 3000]}  # 투수×구종 → 투수×구종×손 쪽으로 당기는 강도
res_loc = grid_search(score_loc, grid_loc)
display(res_loc.head(10).round(4))  # 상위 10개 조합
for k, s in marginal(res_loc, grid_loc).items():  # 파라미터별 민감도
    print(k, s.round(4).to_dict())
best_loc = best_of(res_loc, grid_loc)
print("거친 격자 1등:", best_loc)

,half_life,k_type,k_hand,logloss,top1,H/S/C logloss,H/S/C 정확도,좌표 NLL,평균거리 cm
0,50.0,30,30,2.9280,0.1424,1.0750,0.4027,9.2654,31.9216
1,50.0,100,10,2.9284,0.1433,1.0755,0.4021,9.2613,31.9239
2,50.0,300,10,2.9284,0.1426,1.0756,0.4030,9.2605,31.9211
3,50.0,1000,10,2.9286,0.1429,1.0757,0.4021,9.2604,31.9213
4,50.0,3000,10,2.9287,0.1429,1.0757,0.4018,9.2604,31.9216
5,50.0,100,30,2.9288,0.1435,1.0756,0.4025,9.2625,31.9154
6,50.0,30,10,2.9291,0.1428,1.0754,0.4033,9.2643,31.9379
7,50.0,10,30,2.9295,0.1429,1.0749,0.4033,9.2716,31.9530
8,50.0,300,30,2.9306,0.1427,1.0762,0.4017,9.2630,31.9289
9,50.0,10,10,2.9308,0.1436,1.0755,0.4032,9.2694,31.9616


half_life {'None': 2.9384, 1000: 2.9374, 300: 2.9353, 100: 2.9311, 50: 2.928}
k_type {10: 2.9295, 30: 2.928, 100: 2.9284, 300: 2.9284, 1000: 2.9286, 3000: 2.9287}
k_hand {10: 2.9284, 30: 2.928, 100: 2.9312, 300: 2.9352, 1000: 2.9376, 3000: 2.9385}
거친 격자 1등: {'half_life': 50.0, 'k_type': 30.0, 'k_hand': 30.0}


### 🥇1등🥇
#### 반감 50구, k_type 30, k_hand 30 -> 2019 24칸 log loss 2.9280
#### (베이스라인 300 / 300 / 300: 2.9467)
- 반감기가 가장 민감: 누적 2.938 → 50구 2.928, 짧을수록 좋음 -> 위치도 최근 경기 기록이 중요
  - 50구가 후보 범위 끝 -> 세밀한 격자에서 25까지 넓혀 확인
- k_hand: 10–30이 좋고 3000이면 2.939 -> 투수×구종×타자 손 기록을 강하게 따라가는 게 좋음
- k_type: 30–3000에서 2.928–2.929로 거의 같음 -> 둔감 (k_hand로 다시 당기기 때문에 중간 단계 영향 작음)
- 베이스라인의 "k=300 강한 수축"은 반감기를 누적·300만 시험한 결과 -> 반감기가 짧아져 유효 공 수가 줄면 k도 작은 쪽이 좋음

In [15]:
# 세밀한 격자
grid_loc_f = {k: finer(grid_loc[k], best_loc[k]) for k in grid_loc}
print("세밀한 후보:", grid_loc_f)
res_loc_f = grid_search(score_loc, grid_loc_f)
display(res_loc_f.head(10).round(4))
for k, s in marginal(res_loc_f, grid_loc_f).items():
    print(k, s.round(4).to_dict())
best_loc_f = best_of(res_loc_f, grid_loc_f)

# 선택값 적용 (실전 예측은 이 값 사용)
LOC_HL, K_TYPE, K_HAND = best_loc_f["half_life"], best_loc_f["k_type"], best_loc_f["k_hand"]
base_l = score_loc(300, 300, 300)  # 베이스라인 값 (2019)
print(f"선택: half_life={LOC_HL}, k_type={K_TYPE}, k_hand={K_HAND} | 2019 24칸 log loss {res_loc_f['logloss'].iloc[0]:.4f} "
      f"(베이스라인 {base_l['logloss']:.4f}), 좌표 NLL {res_loc_f['좌표 NLL'].iloc[0]:.4f} (베이스라인 {base_l['좌표 NLL']:.4f})")

세밀한 후보: {'half_life': [25.0, 35.4, 50.0, 70.7, 100.0], 'k_type': [10.0, 17.3, 30.0, 54.8, 100.0], 'k_hand': [10.0, 17.3, 30.0, 54.8, 100.0]}


,half_life,k_type,k_hand,logloss,top1,H/S/C logloss,H/S/C 정확도,좌표 NLL,평균거리 cm
0,35.4,54.8,17.3,2.9270,0.1448,1.0752,0.4048,9.2610,31.8906
1,35.4,30.0,17.3,2.9272,0.1446,1.0751,0.4033,9.2631,31.8987
2,35.4,100.0,17.3,2.9273,0.1440,1.0754,0.4049,9.2601,31.8889
3,25.0,54.8,17.3,2.9273,0.1455,1.0754,0.4048,9.2619,31.8796
4,25.0,30.0,17.3,2.9275,0.1448,1.0752,0.4057,9.2640,31.8870
5,35.4,30.0,30.0,2.9275,0.1433,1.0750,0.4043,9.2652,31.9002
6,35.4,100.0,10.0,2.9275,0.1455,1.0755,0.4034,9.2606,31.8987
7,25.0,100.0,10.0,2.9276,0.1449,1.0756,0.4045,9.2612,31.8854
8,25.0,100.0,17.3,2.9277,0.1446,1.0756,0.4069,9.2610,31.8799
9,35.4,54.8,10.0,2.9277,0.1446,1.0754,0.4033,9.2618,31.9033


half_life {25.0: 2.9273, 35.4: 2.927, 50.0: 2.9278, 70.7: 2.9292, 100.0: 2.9309}
k_type {10.0: 2.929, 17.3: 2.9279, 30.0: 2.9272, 54.8: 2.927, 100.0: 2.9273}
k_hand {10.0: 2.9275, 17.3: 2.927, 30.0: 2.9275, 54.8: 2.9291, 100.0: 2.9311}
선택: half_life=35.4, k_type=54.8, k_hand=17.3 | 2019 24칸 log loss 2.9270 (베이스라인 2.9467), 좌표 NLL 9.2610 (베이스라인 9.2894)


#### 선택: 반감 35.4구, k_type 54.8, k_hand 17.3
#### -> 2019 24칸 log loss 2.9270 (베이스라인 대비 −0.020)
#### ⠀⠀⠀⠀⠀⠀좌표 NLL 9.289 → 9.261, H/S/C 정확도 0.405
- 반감 35.4는 25–50 사이 범위 안쪽 -> 25까지 넓힌 확인 통과
- 세밀한 격자 개선은 0.001뿐, 상위 조합끼리 차이 0.0007 이내 -> 이 근처는 평평함
- 구종(반감 25)과 함께 보면 두 단계 모두 최근 1–3경기 기록이 중요

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- 위치: 9개 구종 각각으로 예측 → 구종 확률로 섞기
- 파라미터는 위 `TYPE_*`·`LOC_*`·`K_*` 현재 값
- 열 규칙은 `fp_common.py` 맨 위 (베이스라인 `per_pitch`와 같은 열)

In [16]:
def predict(q, year):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9)
    Pt = type_online(q, year, TYPE_HL, TYPE_ALPHA, TYPE_HAND_K)

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [17]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR), predict(q_test, fc.TEST_YEAR)], ignore_index=True)
print(pp.shape)

(26073, 48)


## 5. 평가

- 채점은 `fc.evaluate` (compare와 같은 기준), 베이스라인과 나란히
- 2020은 확인용 → 2020을 보고 파라미터 다시 고르지 않음

In [18]:
base_pp = pd.read_parquet("results/baseline/per_pitch.parquet")  # 베이스라인 공별 예측
summary = pd.DataFrame({"검증 2019": fc.evaluate(pp, fp, fc.VAL_YEAR),
                        "평가 2020": fc.evaluate(pp, fp, fc.TEST_YEAR)})
base_sum = pd.DataFrame({"검증 2019": fc.evaluate(base_pp, fp, fc.VAL_YEAR),
                         "평가 2020": fc.evaluate(base_pp, fp, fc.TEST_YEAR)})
display(pd.concat({MODEL: summary, "baseline": base_sum, "차이": summary - base_sum}, axis=1).round(4))

model_01          baseline               차이        
                     검증 2019  평가 2020  검증 2019  평가 2020 검증 2019 평가 2020
구종 logloss            1.0982   1.0956   1.1037   1.1018 -0.0055 -0.0062
구종 top1               0.5258   0.5233   0.5305   0.5244 -0.0047 -0.0010
구종 top2               0.8157   0.8162   0.8104   0.8111  0.0053  0.0051
구종 top3               0.9331   0.9359   0.9318   0.9324  0.0013  0.0035
구종 macro F1           0.2476   0.2857   0.2359   0.2576  0.0117  0.0281
24칸 logloss (실전)      2.9780   2.9708   2.9928   2.9791 -0.0148 -0.0082
24칸 top1              0.1289   0.1339   0.1280   0.1328  0.0010  0.0011
24칸 top3              0.2841   0.2850   0.2781   0.2828  0.0060  0.0021
H/S/C logloss         1.0771   1.0783   1.0803   1.0800 -0.0032 -0.0017
H/S/C 정확도             0.3947   0.3902   0.3926   0.3883  0.0022  0.0020
24칸 logloss (실제 구종)   2.9270   2.9226   2.9467   2.9403 -0.0197 -0.0177
구종×칸 logloss          4.0252   4.0182   4.0504   4.0419 -0.0252 -0.0237
좌표 평균거리 cm           32.7934  31.9009  32.9864  31.9869 -0.1930 -0.0860
좌표 NLL                9.3235   9.2708   9.3459   9.2882 -0.0224 -0.0174

- ① 구종: 2020 log loss 1.1018 → 1.0956 (−0.006), top2 +0.005, macro F1 +0.028, top1 거의 같음 -> 2019에서 고른 개선이 2020에도 이어짐
- ② 위치: 2020 24칸 log loss 실제 구종 2.940 → 2.923 (−0.018), 실전 2.979 → 2.971 (−0.008)
  - 기준선(리그 24칸 비율 2.995) 대비 차이 0.016 → 0.024로 커짐, 그래도 작음
  - 실전 개선이 실제 구종 기준의 절반 -> 당김이 약해져 투수가 드물게 던지는 구종의 위치 예측이 흔들렸을 가능성, 확인 필요
  - H/S/C 정확도 0.388 → 0.390 -> 구분력은 여전히 거의 없음
- 구종×칸 log loss 4.042 → 4.018 (−0.024)
- 차이가 의미 있는지는 저장 후 compare 부트스트랩으로 확인

## 6. 저장

- `results/model_01/` → `compare.ipynb` 실행하면 자동 포함 (부트스트랩 판정은 compare에서)

In [19]:
fc.save_results(MODEL, pp, {"summary": summary})  # per_pitch.parquet + summary.csv

저장: results\model_01 ['per_pitch.parquet', 'summary.csv']


## 7. 진단: 실전 24칸 개선이 실제 구종 기준의 절반인 이유

- 2020: 실제 구종 기준 2.940 → 2.923 (−0.018), 실전 2.979 → 2.971 (−0.008)
- 확인할 것
  1. 개선이 구종 파라미터에서 왔나, 위치 파라미터에서 왔나
  2. 구종을 몰라서 잃는 몫(실전 − 실제 구종)이 커졌나 -> 구종별 위치 분포가 서로 더 달라졌는지
  3. 어떤 공에서 실제 구종 기준 개선이 실전으로 안 이어지나 (실제 구종 확률, 투수×구종×손 기록 수)

In [19]:
P_BASE = {"type": (40, 1.0, 20), "loc": (300, 300, 300)}  # 베이스라인 파라미터
P_M01 = {"type": (TYPE_HL, TYPE_ALPHA, TYPE_HAND_K), "loc": (LOC_HL, K_TYPE, K_HAND)}  # 위에서 고른 값
nll = lambda p: -np.log(np.clip(p, 1e-12, None))

def zone_parts(q, year):  # 구종 확률 {base, m01} (n, 9), 구종별 24칸 확률 {base, m01} (n, 9, 24), 정답 번호
    Pt = {k: type_online(q, year, *p["type"]) for k, p in [("base", P_BASE), ("m01", P_M01)]}
    PZ = {k: np.stack([loc_given_type(q, year, t, *p["loc"])[0] for t in CLASSES], 1) for k, p in [("base", P_BASE), ("m01", P_M01)]}
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()
    return Pt, PZ, yt, yz

parts = {fc.VAL_YEAR: zone_parts(q_val, fc.VAL_YEAR), fc.TEST_YEAR: zone_parts(q_test, fc.TEST_YEAR)}

In [20]:
# 1. 구종 모델 × 위치 모델 조합별 실전 24칸 log loss (+ 실제 구종 기준)
for year, (Pt, PZ, yt, yz) in parts.items():
    i = np.arange(len(yt))
    tab = pd.DataFrame({f"구종 {tk}": [nll(np.einsum("nt,ntz->nz", Pt[tk], PZ[lk])[i, yz]).mean() for lk in PZ] for tk in Pt},
                       index=[f"위치 {lk}" for lk in PZ])  # 실전 = Σ P(구종)·P(칸|구종)
    tab["실제 구종"] = [nll(PZ[lk][i, yt, yz]).mean() for lk in PZ]
    print(year); display(tab.round(4))

2019


,구종 base,구종 m01,실제 구종
위치 base,2.9928,2.9934,2.9467
위치 m01,2.9774,2.9780,2.9270


2020


,구종 base,구종 m01,실제 구종
위치 base,2.9791,2.9793,2.9403
위치 m01,2.9705,2.9708,2.9226


### 🔍 실전 개선은 거의 전부 위치 파라미터에서
- 구종만 m01로 바꾸면 실전 24칸은 거의 그대로 (2020 2.9791 → 2.9793, 위치 m01에서도 2.9705 → 2.9708)
  - 구종 log loss가 좋아져도 24칸에는 안 이어짐
- 위치만 m01로 바꾸면 실전 2.9791 → 2.9705 (−0.009), 실제 구종 기준 −0.018

In [21]:
# 2. 구종을 몰라서 잃는 몫 = 실전 − 실제 구종 (구종 모델은 m01로 고정, 위치 모델만 바꿈)
# 모델이 기대한 몫 = Σ P(구종)·KL(P(칸|구종) || 섞은 분포) = 모델 안에서 구종과 칸의 상호정보 -> 구종별 위치 분포가 서로 다를수록 큼
rows = {}
for year, (Pt, PZ, yt, yz) in parts.items():
    i = np.arange(len(yt))
    for lk in PZ:
        Pmix = np.einsum("nt,ntz->nz", Pt["m01"], PZ[lk])  # 실전 24칸 확률
        kl = (Pt["m01"] * (PZ[lk] * (np.log(np.clip(PZ[lk], 1e-12, None)) - np.log(Pmix[:, None, :]))).sum(2)).sum(1)
        rows[(year, f"위치 {lk}")] = {"실전 − 실제 구종": (nll(Pmix[i, yz]) - nll(PZ[lk][i, yt, yz])).mean(),
                                      "모델이 기대한 몫": kl.mean()}
display(pd.DataFrame(rows).T.round(4))

실전 − 실제 구종  모델이 기대한 몫
2019 위치 base      0.0467     0.0401
     위치 m01       0.0510     0.0576
2020 위치 base      0.0389     0.0417
     위치 m01       0.0482     0.0581

### 🔍 구종을 몰라서 잃는 몫이 커짐
#### 2020: 0.039 → 0.048, 모델이 기대한 몫 0.042 → 0.058
- 베이스라인은 k=300으로 강하게 당겨 구종별 위치 분포가 리그 쪽으로 서로 비슷함 -> 구종을 몰라도 잃을 게 적었음
- m01은 투수×구종×손 기록을 많이 따라감 -> 구종별 위치 분포가 서로 더 다름 -> 구종을 틀리면 잃는 몫도 커짐
- 실제 구종 기준 개선 −0.018 중 약 −0.009가 이 몫 증가로 상쇄 -> 절반만 실전으로
- m01은 실제로 잃은 몫(0.048)이 모델이 기대한 몫(0.058)보다 작음 -> 구종별 분포 차이를 조금 크게 잡았을 수 있음 (베이스라인은 반대)

In [22]:
# 3. 그룹별 개선 (위치 m01 − 위치 base, 구종은 m01로 고정): 실제 구종 기준 개선이 실전으로 이어지는지
def group_gain(year, by):  # by: 공마다 그룹 이름 (배열), 반환: 그룹별 공 수·개선·전체 개선에 기여하는 몫
    Pt, PZ, yt, yz = parts[year]
    i = np.arange(len(yt))
    mix = {lk: nll(np.einsum("nt,ntz->nz", Pt["m01"], PZ[lk])[i, yz]) for lk in PZ}
    d = pd.DataFrame({"g": by, "실제 구종 개선": nll(PZ["m01"][i, yt, yz]) - nll(PZ["base"][i, yt, yz]),
                      "실전 개선": mix["m01"] - mix["base"]})
    t = d.groupby("g", observed=True).agg(공수=("실전 개선", "size"), **{c: (c, "mean") for c in ["실제 구종 개선", "실전 개선"]})
    t["실전 개선 몫"] = t["실전 개선"] * t["공수"] / len(d)  # 합 = 전체 실전 개선
    return t

for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    Pt, PZ, yt, yz = parts[year]
    p_true = Pt["m01"][np.arange(len(yt)), yt]  # 실제 구종에 준 확률
    _, N = fc.lookup(fc.prior_table(fp, K2, ["x"], None), q, K2, ["x"])  # 투수×구종×손 전날까지 누적 공 수 (반감기 없이)
    print(year, "· 실제 구종에 준 확률"); display(group_gain(year, pd.cut(p_true, [0, .05, .2, .5, 1.0])).round(4))
    print(year, "· 투수×구종×손 기록 수"); display(group_gain(year, pd.cut(N[:, 0], [-1, 0, 30, 100, 300, 1e9],
                                                                          labels=["0", "1-30", "31-100", "101-300", "300+"])).round(4))

2019 · 실제 구종에 준 확률


,공수,실제 구종 개선,실전 개선,실전 개선 몫
g,,,,
"(0.0, 0.05]",350,-0.0529,0.0025,0.0001
"(0.05, 0.2]",2000,-0.0104,0.0017,0.0003
"(0.2, 0.5]",6547,-0.0213,-0.0158,-0.0077
"(0.5, 1.0]",4561,-0.0188,-0.0237,-0.0080


2019 · 투수×구종×손 기록 수


,공수,실제 구종 개선,실전 개선,실전 개선 몫
g,,,,
0,89,0.0002,0.0117,0.0001
1-30,1620,-0.0250,-0.0050,-0.0006
31-100,3616,-0.0189,-0.0101,-0.0027
101-300,5672,-0.0190,-0.0161,-0.0068
300+,2461,-0.0197,-0.0292,-0.0053


2020 · 실제 구종에 준 확률


,공수,실제 구종 개선,실전 개선,실전 개선 몫
g,,,,
"(0.0, 0.05]",348,-0.0934,-0.0030,-0.0001
"(0.05, 0.2]",1764,-0.0327,0.0023,0.0003
"(0.2, 0.5]",6238,-0.0182,-0.0091,-0.0045
"(0.5, 1.0]",4265,-0.0046,-0.0123,-0.0041


2020 · 투수×구종×손 기록 수


,공수,실제 구종 개선,실전 개선,실전 개선 몫
g,,,,
0,34,-0.0120,-0.0456,-0.0001
1-30,729,-0.0872,0.0068,0.0004
31-100,2355,-0.0268,-0.0081,-0.0015
101-300,5826,-0.0113,-0.0089,-0.0041
300+,3671,-0.0082,-0.0106,-0.0031


### 🔍 개선이 실제 구종 확률이 낮은 공에 몰림
- 실제 구종 확률 0.2 이하 공(약 17%): 실제 구종 기준으로는 크게 좋아짐 (2020 −0.03 ~ −0.09), 실전은 거의 그대로 (−0.003 ~ +0.002)
  - 투수가 드물게 던지는 구종의 위치를 더 잘 맞히게 됐지만, 실전에서는 그 구종에 준 확률이 작아 거의 안 쓰임
- 기록 1–30개 그룹도 같음: 실제 구종 기준 −0.087, 실전 +0.007 (2020)
  - "드문 구종 위치 예측이 흔들림" 가설은 아님 -> 드문 구종의 위치는 오히려 가장 많이 좋아짐
- 실제 구종 확률 0.5 넘는 공: 실전 개선이 실제 구종 기준보다 큼 (2020 −0.012 vs −0.005)

### 결론
- 오류가 아니라 구조 때문: 위치 모델이 구종을 더 구분하게 됨 -> 남은 개선은 구종을 맞혀야 실전으로 나옴
- -> 2단계 구종 개선의 값이 커짐 (구종과 칸의 상호정보 0.042 → 0.058)
- 지금 따로 고칠 것은 없음, 모델이 기대한 몫이 실제보다 큰 점은 위치 분포 퍼짐(분산) 확인 때 같이 보기